**姓名 / Name:** _____

**学号 / EID:** _____

# 教程 1：使用 KNN 和朴素贝叶斯进行文本分类 / Tutorial 1: Text Document Classification with KNN and Naive Bayes

在本教程中，你将使用朴素贝叶斯分类器对文本文档进行分类。我们将使用名为“20 Newsgroups”的数据集，它包含约 20,000 篇按 20 个类别组织的新闻组帖子。

In this tutorial you will classify text documents using Naive Bayes classifers. We will be working with the dataset called "20 Newsgroups", which is a collection of 20,000 newsgroup posts organized into 20 categories.

## 1. 加载 20 Newsgroups 数据集 / Loading the 20 Newsgroups Dataset
数据集名称为“20 Newsgroups”。下面是官方网站上的说明：

The dataset is called "20 Newsgroups". Here is the official description, quoted from the website:
> The 20 Newsgroups data set is a collection of approximately 20,000 newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups. To the best of our knowledge, it was originally collected by Ken Lang, probably for his paper "Newsweeder: Learning to filter netnews," though he does not explicitly mention this collection. The 20 newsgroups collection has become a popular data set for experiments in text applications of machine learning techniques, such as text classification and text clustering.

**注意 / Note**: 本教程推荐使用 **scikit-learn 1.4.2**。其他版本可能会出现兼容性问题。

In [ ]:
# 首先初始化 Python。运行下面的单元。 / First we need to initialize Python. Run the below cell.
%matplotlib inline
import IPython.core.display
# 设置输出图像格式（Chrome 效果最佳）。 / Setup output image format (Chrome works best).
IPython.core.display.set_matplotlib_formats("svg")
import matplotlib.pyplot as plt
import matplotlib
from numpy import *
from sklearn import *
from scipy import stats
random.seed(100)

- 将文件 `20news-bydate_py3.pkz` 放在与当前 ipynb 文件相同的目录下，**不要解压**。
- Put the file `20news-bydate_py3.pkz` into the same directory as this ipynb file. **Do not unzip the file**.

- 从数据集中提取 4 个类别：`'alt.atheism'`, `'talk.religion.misc'`, `'comp.graphics'`, `'sci.space'`。
- Extract 4 classes (`'alt.atheism'`, `'talk.religion.misc'`, `'comp.graphics'`, `'sci.space'`) from the dataset.

In [ ]:
# 去除文本中的页眉、页脚和引用。 / Strip away headers, footers, and quotes from the text.
removeset = ('headers', 'footers', 'quotes')

# 仅使用 4 个类别。 / Only use 4 categories.
cats      = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']

# 加载训练集和测试集。 / Load the training and testing sets.
newsgroups_train = datasets.fetch_20newsgroups(subset='train',
                           remove=removeset, categories=cats, data_home='./')
newsgroups_test  = datasets.fetch_20newsgroups(subset='test',
                           remove=removeset, categories=cats, data_home='./')

- 检查是否正确读取了全部数据。训练集应有 2034 篇文档，测试集应有 1353 篇文档。
- Check if we got all the data. The training set should have 2034 documents, and the test set should have 1353 documents.

In [ ]:
print("训练集大小 / Training set size:", len(newsgroups_train.data))
print("测试集大小 / Testing set size: ",  len(newsgroups_test.data))
print("类别名称 / Target names:", newsgroups_train.target_names)

- 统计每个类别中的样本数量。`newsgroups_train.target` 是一个类别标签数组（0 到 3），`newsgroups_train.target[i]` 表示第 `i` 个文档所属的类别。
- Count the number of examples in each class. `newsgroups_train.target` is an array of class values (0 through 3), and `newsgroups_train.target[i]` is the class of the i-th document.

In [ ]:
print("各类别样本数 / Class counts")
for i in [0, 1, 2, 3]:
    print("{:20s}: {}".format(newsgroups_train.target_names[i], sum(newsgroups_train.target == i)))

- 展示文档内容。`newsgroups_train.data` 是字符串列表，`newsgroups_train.data[i]` 表示第 `i` 个文档。
- Show the documents. `newsgroups_train.data` is a list of strings, and `newsgroups_train.data[i]` is the i-th document.

In [ ]:
for i in [0, 1, 2 ,3]:
    print("--- 文档 / document {} (class={}) ---".format(
        i, newsgroups_train.target_names[newsgroups_train.target[i]]))
    print(newsgroups_train.data[i])

**提示 / Tip：** 在完成本教程时，你可以额外添加代码单元，这样可以避免重复运行已经完成的部分（例如先训练分类器，再测试分类器）。

**Tip:** while you do the tutorial, it is okay to make additional code cells in the file. This will allow you to avoid re-running code (like training a classifier, then testing a classifier).

## 2. 从文本中提取特征 / Extracting Features from Text Files
为了在文本文档上进行机器学习，我们首先需要将文本内容转换为数值特征向量。

接下来我们将介绍两种基础的文本表示方法：One-hot 编码、词袋模型（Bag of Words）和 TF-IDF。更多特征提取函数可参考：https://scikit-learn.org/stable/modules/classes.html#module-sklearn.feature_extraction

In order to perform machine learning on text documents, we first need to turn the text content into numerical feature vectors.

Next, we will introduce two basic text representation methods: One-hot encoding, Bag of Words, and TF-IDF. More feature vector extraction functions, please refer to https://scikit-learn.org/stable/modules/classes.html#module-sklearn.feature_extraction

### One-hot 编码 / One-hot Encoding
- 每个单词都会被分配一个索引，并用 one-hot 向量表示。
- Each word is coded with an index, which is represented by one-hot.

> 中文示例 / Chinese example:
> John 喜欢看电影。Mary 也喜欢。
> John 也喜欢看足球比赛。

> English example:
> John likes to watch movies. Mary likes too.
> John also likes to watch football games.

如果我们需要表示上面两个句子中的单词，可以按如下方式进行编码。

If we need to represent the words in the above two sentences, you can encode the words as follows:

> 词典 / Vocabulary
> {"John": 1, "likes": 2, "to": 3, "watch": 4, "movies": 5, "also": 6, "football": 7, "games": 8, "Mary": 9, "too": 10}

我们可以使用 one-hot 方法对每个单词进行编码。

We can encode each word using the one-hot method.

> John: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

> likes: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

> ...

#### 但当语料规模很大时，这种表示方式并不实用。 / However, this text representation method is impractical when the scale of corpus becomes large.

### 词袋模型 / Bag of Words
- 词汇表中单词的索引与其在整个训练语料中的统计特征相关，这里通常使用词频。
- The index value of a word in the vocabulary is linked to its frequency in the whole training corpus.

> John likes to watch movies. Mary likes too.  -->> [1, 2, 1, 1, 1, 0, 0, 0, 1, 1]

> John also likes to watch football games.     -->> [1, 1, 1, 1, 0, 1, 1, 1, 0, 0]

`sklearn.feature_extraction.text.CountVectorizer` 实现了 **Bag of Words** 方法，可以将文本集合转换为词频矩阵。该实现会使用 **scipy.sparse.coo_matrix** 的稀疏表示，只保存非零部分，从而节省内存。

The **sklearn.feature_extraction.text.CountVectorizer** implements the `Bag of Words` method that converts a collection of text documents to a matrix of token counts. This implementation produces a sparse representation of the counts using **scipy.sparse.coo_matrix** to save memory by only storing the non-zero parts of the feature vectors in memory.

### 词频-逆文档频率 / Term Frequency - Inverse Document Frequency (TF-IDF)

在词袋模型中，我们可以得到文本的向量表示。但在实际应用中，不同单词对文本内容的重要性不同，因此引入了 TF-IDF 模型。

In the word bag model, we can get the vector representation of this text. However, in the face of the diversity of text, each word has different weight to the content of text in practical application, so we introduce tf-idf model.

##### TF（词频 / Term Frequency）
在词频 $tf(t, d)$ 的情况下，最简单的选择是使用某个词项 $t$ 在文档 $d$ 中出现的原始次数。如果用 $f_{t,d}$ 表示该原始次数，则最简单的 tf 方案为 $tf(t,d)=f_{t,d}$。

$tf_{t, d} = n_{t, d}/\sum_k n_{t, d}$

上式分子表示单词在文档 $d$ 中的出现次数，分母表示文档 $d$ 中所有单词出现次数之和。

##### IDF（逆文档频率 / Inverse Document Frequency）
逆文档频率衡量一个单词提供了多少信息，即它在所有文档中是常见还是稀有。其定义为包含该词项文档数的反比并取对数：

$idf(t, D) = log\frac{N}{|\{ d\in D:t \in d \}|}$

其中：
- $N$：语料库中的文档总数，$N=|D|$
- $|\{ d\in D:t \in d \}|$：包含词项 $t$ 的文档数。如果该词不在语料中，会造成除零，因此常将分母改为 $1 + |\{ d\in D:t \in d \}|$。

因此 tf-idf 的计算方式为：
$tfidf(t, d, D) = tf(t, d) * idf(t, D)$

Both tf and tf-idf can be computed as follows using **sklearn.feature_extraction.text.TfidfTransformer**.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = ['This is the first document.',
	'This document is the second document.',
	'And this is the third one.',
	'Is this the first document?']

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print("特征词 / Feature names:")
print(vectorizer.get_feature_names_out())
print("词频矩阵 / Count matrix:")
print(X.toarray())


请基于训练数据创建词汇表，然后使用 **sklearn.feature_extraction.text.CountVectorizer** 为训练集和测试集构建文档向量。词汇表大小可自行决定。

Create the vocabulary from the training data. Then use **sklearn.feature_extraction.text.CountVectorizer** to build the document vectors for the training and testing sets. You can decide how many words you want in the vocabulary.

In [ ]:
# 基于训练语料构建词袋表示。 / Build a bag-of-words representation from the training corpus.
cntvect = feature_extraction.text.CountVectorizer(stop_words='english')
Xtrain_counts = cntvect.fit_transform(newsgroups_train.data)
Xtest_counts = cntvect.transform(newsgroups_test.data)

print('训练矩阵形状 / Training matrix shape:', Xtrain_counts.shape)
print('测试矩阵形状 / Testing matrix shape: ', Xtest_counts.shape)
print('词汇表大小 / Vocabulary size:', len(cntvect.get_feature_names_out()))

## 3. K 近邻（KNN）/ K Nearest Neighbor (KNN)
训练一个 KNN 模型。使用交叉验证选择最优的 K 值，然后给出训练集和测试集上的准确率。

Let's train a K Nearest Neighbor (KNN) model. Using cross-validation to select the best K parameter. Then, showing the accuracy of training and testing set.

In [ ]:
paramgrid = {'n_neighbors': [3, 5, 7]}
knn_search = model_selection.GridSearchCV(
    neighbors.KNeighborsClassifier(),
    param_grid=paramgrid,
    cv=10,
    n_jobs=-1
)
knn_search.fit(Xtrain_counts, newsgroups_train.target)

print('最佳参数 / Best params:', knn_search.best_params_)
print('训练准确率 / Training accuracy:', knn_search.score(Xtrain_counts, newsgroups_train.target))
print('测试准确率 / Testing accuracy: ', knn_search.score(Xtest_counts, newsgroups_test.target))

## 4. 伯努利朴素贝叶斯 / Bernoulli Naive Bayes
从训练集学习一个 Bernoulli Naive Bayes 模型。它在测试集上的预测准确率是多少？尝试不同参数（如 `alpha`、`max_features` 等）来获得更好的性能。

Learn a Bernoulli Naive Bayes model from the training set. What is the prediction accuracy on the test set? Try different parameters (`alpha`, `max_features`, etc.) to get the best performance.

In [ ]:
Xtrain_binary = (Xtrain_counts > 0).astype(int)
Xtest_binary = (Xtest_counts > 0).astype(int)

bernoulli_alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
best_bernoulli = None
for alpha in bernoulli_alphas:
    model = naive_bayes.BernoulliNB(alpha=alpha)
    model.fit(Xtrain_binary, newsgroups_train.target)
    test_acc = model.score(Xtest_binary, newsgroups_test.target)
    if best_bernoulli is None or test_acc > best_bernoulli[0]:
        best_bernoulli = (test_acc, alpha, model)

bmodel = best_bernoulli[2]
print('最佳 alpha / Best alpha:', best_bernoulli[1])
print('训练准确率 / Training accuracy:', bmodel.score(Xtrain_binary, newsgroups_train.target))
print('测试准确率 / Testing accuracy: ', best_bernoulli[0])

每个类别中最有信息量的词是什么？运行下面的代码。
注意：`model.feature_log_prob_[i]` 会索引到第 `i` 个类别对应的分数。

What are the most informative words for each category? Run the below code.

Note: `model.feature_log_prob_[i]` will index the scores for the i-th class.

In [ ]:
# 获取词语名称。 / Get the word names.
fnames = asarray(cntvect.get_feature_names_out())
for i,c in enumerate(newsgroups_train.target_names):
    tmp = argsort(bmodel.feature_log_prob_[i])[-10:]
    print("类别 / class", c)
    for t in tmp:
        print("    {:9s} ({:.5f})".format(fnames[t], bmodel.feature_log_prob_[i][t]))

## 5. 多项式朴素贝叶斯模型 / Multinomial Naive Bayes Model
现在使用文档的 TF-IDF 表示来学习一个多项式朴素贝叶斯模型。同样尝试不同的参数值，以提升测试准确率。

Now learn a multinomial Naive Bayes model using the TF-IDF representation for the documents. Again try different parameter values to improve the test accuracy.

In [ ]:
tfidf_transformer = feature_extraction.text.TfidfTransformer(use_idf=True, norm='l2')
Xtrain_tfidf = tfidf_transformer.fit_transform(Xtrain_counts)
Xtest_tfidf = tfidf_transformer.transform(Xtest_counts)

multinomial_alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
best_multinomial = None
for alpha in multinomial_alphas:
    model = naive_bayes.MultinomialNB(alpha=alpha)
    model.fit(Xtrain_tfidf, newsgroups_train.target)
    test_acc = model.score(Xtest_tfidf, newsgroups_test.target)
    if best_multinomial is None or test_acc > best_multinomial[0]:
        best_multinomial = (test_acc, alpha, model)

mmodel_tf = best_multinomial[2]
print('最佳 alpha / Best alpha:', best_multinomial[1])
print('训练准确率 / Training accuracy:', mmodel_tf.score(Xtrain_tfidf, newsgroups_train.target))
print('测试准确率 / Testing accuracy: ', best_multinomial[0])

多项式模型中最有信息量的特征是什么？运行下面的代码。

What are the most informative features for Multinomial model? Run the below code.

In [ ]:
# 获取词语名称。 / Get the word names.
fnames = asarray(cntvect.get_feature_names_out())
for i,c in enumerate(newsgroups_train.target_names):
    tmp = argsort(mmodel_tf.feature_log_prob_[i])[-10:]
    print("类别 / class", c)
    for t in tmp:
        print("    {:9s} ({:.5f})".format(fnames[t], mmodel_tf.feature_log_prob_[i][t]))

In [ ]:
# 此双语版本用于本地使用，不需要挂载 Google Drive。
# This bilingual version is intended for local use, so no Google Drive mount is required.


TF-IDF 多项式模型和 Bernoulli 模型中最有信息量的词有什么不同？

How do the most informative words differ between the TF-IDF multinomial model and the Bernoulli model?

- Bernoulli 模型更关注某个词是否出现，因此更容易突出 `god`、`people`、`graphics`、`space`、`thanks` 这类较宽泛的类别标记词。
- The Bernoulli model highlights whether a word appears at all, so it tends to surface broader class markers such as `god`, `people`, `graphics`, `space`, and `thanks`.
- TF-IDF 多项式模型会更强调在某一类别中更具区分性的词，因此更容易得到 `atheism`、`orbit`、`shuttle`、`jesus`、`windows`、`image` 等更具体的词。
- The TF-IDF multinomial model gives higher weight to words that are especially distinctive for one class, so its top words are usually more specific, such as `atheism`, `orbit`, `shuttle`, `jesus`, `windows`, and `image`.

## 6. 平滑参数的影响 / Effect of Smoothing
平滑（正则化）参数会显著影响模型性能。请使用 Multinomial + TF-IDF 模型，绘制准确率随 `alpha` 变化的曲线。对于每个 `alpha`，都需要重新训练一个模型。哪个 `alpha` 的结果最好？

The smoothing (regularization) parameter has a big effect on the performance. Using the Multinomial TF-IDF models, make a plot of accuracy versus different values of alpha. For each alpha, you need to train a new model. Which alpha value yields the best result?

In [ ]:
alphas = logspace(-5, 0, 50)
alpha_scores = []
for alpha in alphas:
    model = naive_bayes.MultinomialNB(alpha=float(alpha))
    model.fit(Xtrain_tfidf, newsgroups_train.target)
    alpha_scores.append(model.score(Xtest_tfidf, newsgroups_test.target))

plt.figure(figsize=(6, 4))
plt.semilogx(alphas, alpha_scores, marker='o', linewidth=1)
plt.xlabel('alpha / 平滑参数')
plt.ylabel('test accuracy / 测试准确率')
plt.title('Multinomial NB with TF-IDF: alpha vs accuracy / 平滑参数与准确率')
plt.grid(True)

best_alpha_idx = argmax(alpha_scores)
print('最佳 alpha / Best alpha:', alphas[best_alpha_idx])
print('最佳准确率 / Best accuracy:', alpha_scores[best_alpha_idx])

## 7. 词汇表大小的影响 / Effect of Vocabulary Size
词汇表大小也会影响准确率。请再绘制一张准确率随词汇表大小变化的曲线。哪个词汇表大小的效果最好？

The vocabulary size also affects the accuracy. Make another plot of accuracy versus vocabulary size. Which vocabulary size yields the best result?

In [ ]:
maxfeatures = linspace(100, 26577, 20).astype(int)
vocab_scores = []
for max_features in maxfeatures:
    vect = feature_extraction.text.CountVectorizer(stop_words='english', max_features=int(max_features))
    Xtrain_vocab = vect.fit_transform(newsgroups_train.data)
    Xtest_vocab = vect.transform(newsgroups_test.data)
    tfidf = feature_extraction.text.TfidfTransformer(use_idf=True, norm='l2')
    Xtrain_vocab_tfidf = tfidf.fit_transform(Xtrain_vocab)
    Xtest_vocab_tfidf = tfidf.transform(Xtest_vocab)
    model = naive_bayes.MultinomialNB(alpha=best_multinomial[1])
    model.fit(Xtrain_vocab_tfidf, newsgroups_train.target)
    vocab_scores.append(model.score(Xtest_vocab_tfidf, newsgroups_test.target))

plt.figure(figsize=(6, 4))
plt.plot(maxfeatures, vocab_scores, marker='o', linewidth=1)
plt.xlabel('max_features / 词汇表大小')
plt.ylabel('test accuracy / 测试准确率')
plt.title('Multinomial NB with TF-IDF: vocabulary size vs accuracy / 词汇表大小与准确率')
plt.grid(True)

best_vocab_idx = argmax(vocab_scores)
print('最佳词汇表大小 / Best vocabulary size:', maxfeatures[best_vocab_idx])
print('最佳准确率 / Best accuracy:', vocab_scores[best_vocab_idx])